# 🧠 Skin Type + Skin Problem Classifier
Train a multi-task model to predict skin type and skin problems from facial images.
- Face detection via OpenCV
- Model: Shared MobileNetV2 backbone with 2 output heads
- Output: Percentages for skin type and problems

In [ ]:
# 📦 Imports
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import img_to_array, load_img

In [ ]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
tf.debugging.set_log_device_placement(True)

In [ ]:
# 📂 Set Dataset Paths (Kaggle Datasets section)
SKIN_TYPE_DIR = '/kaggle/input/oily-dry-and-normal-skin-types-dataset/Oily-Dry-Skin-Types/train'
SKIN_PROBLEM_DIR = '/kaggle/input/skin-defects-acne-redness-and-bags-under-the-eyes'
SKIN_PROBLEM_CSV = os.path.join(SKIN_PROBLEM_DIR, 'skin_defects.csv')
IMG_SIZE = 224

In [ ]:
# 🖼️ Utility: Face detection and cropping using OpenCV
def detect_and_crop_face(image_path):
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)
    if len(faces) == 0:
        return None
    (x, y, w, h) = faces[0]
    face = img[y:y+h, x:x+w]
    face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
    return face / 255.0

In [ ]:
# 🧹 Load and preprocess skin type data
skin_type_data = []
labels_type = os.listdir(SKIN_TYPE_DIR)
for label in labels_type:
    path = os.path.join(SKIN_TYPE_DIR, label)
    for fname in os.listdir(path):
        fpath = os.path.join(path, fname)
        face = detect_and_crop_face(fpath)
        if face is not None:
            skin_type_data.append((face, label))

X_type, y_type = zip(*skin_type_data)
X_type = np.array(X_type)
label_map_type = {l: i for i, l in enumerate(sorted(set(y_type)))}
y_type = to_categorical([label_map_type[y] for y in y_type])

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelBinarizer
import os

# Load CSV
df = pd.read_csv(SKIN_PROBLEM_CSV)

# Initialize label binarizer and container
lb = LabelBinarizer()
skin_prob_data = []

# Fit binary labels from the 'type' column
y_bin = lb.fit_transform(df['type'])  # Use the correct column name in your CSV

# Iterate over each row to process images
for idx, row in df.iterrows():
    img_rel_path = row['front'].lstrip('/')  # Adjust path if needed
    img_path = os.path.join(SKIN_PROBLEM_DIR, 'files', img_rel_path)

    face = detect_and_crop_face(img_path)
    if face is not None:
        skin_prob_data.append((face, y_bin[idx]))

# Convert to NumPy arrays
X_prob, y_prob = zip(*skin_prob_data)
X_prob = np.array(X_prob)
y_prob = np.array(y_prob)


In [ ]:
# 🧹 Load and preprocess skin problem data
df = pd.read_csv(SKIN_PROBLEM_CSV)
skin_prob_data = []
mlb = MultiLabelBinarizer()
df['labels'] = df['Class'].apply(lambda x: x.split(','))
Y_bin = mlb.fit_transform(df['labels'])

for idx, row in df.iterrows():
    img_path = os.path.join(SKIN_PROBLEM_DIR, row['Filename'])
    face = detect_and_crop_face(img_path)
    if face is not None:
        skin_prob_data.append((face, Y_bin[idx]))

X_prob, y_prob = zip(*skin_prob_data)
X_prob = np.array(X_prob)
y_prob = np.array(y_prob)

In [ ]:
# 🧩 Combine datasets (only matching sample size for simplicity here)
min_len = min(len(X_type), len(X_prob))
X = X_type[:min_len]
y_type_final = y_type[:min_len]
y_prob_final = y_prob[:min_len]

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
df['labels'] = df['type'].apply(lambda x: x.split(','))
mlb.fit(df['labels'])

In [ ]:
# 🏗️ Model definition (multi-output CNN)
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
base = MobileNetV2(include_top=False, weights='imagenet', input_tensor=inputs)
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

# Output 1: Skin Type (3 classes)
out_type = layers.Dense(3, activation='softmax', name='skin_type')(x)

# Output 2: Skin Problems (multi-label)
out_prob = layers.Dense(len(mlb.classes_), activation='sigmoid', name='skin_problem')(x)

model = Model(inputs, outputs=[out_type, out_prob])
model.compile(
    optimizer='adam',
    loss={'skin_type': 'categorical_crossentropy', 'skin_problem': 'binary_crossentropy'},
    metrics={'skin_type': 'accuracy', 'skin_problem': 'accuracy'}
)
model.summary()

In [ ]:
model.save('multitask_skin_model.h5')


## 📊 Final Evaluation Summary
- **Skin Type Classification (Single-label)**: Accuracy, Confusion Matrix, F1-Score
- **Skin Problem Detection (Multi-label)**: Precision, Recall, F1-score per issue, Hamming Loss
- **Calibration**: Confidence distribution
- **Robustness**: Test with augmented images (optional)
- **Deployment Readiness**: Inference speed and model size (optional)


In [ ]:
# 🔍 Evaluate Skin Type Classification (Single-label)
from sklearn.metrics import classification_report, confusion_matrix

# Assuming y_true_type and y_pred_type are available
y_pred_type = model.predict(X_test_skin_type)
y_pred_type_classes = y_pred_type.argmax(axis=1)

print(classification_report(y_true_type, y_pred_type_classes))
print(confusion_matrix(y_true_type, y_pred_type_classes))

In [ ]:
# 🔍 Evaluate Skin Problem Detection (Multi-label)
from sklearn.metrics import classification_report, hamming_loss

# Assuming y_true_issues and y_pred_issues are available
y_pred_issues = model.predict(X_test_skin_issues) > 0.5

print(classification_report(y_true_issues, y_pred_issues, zero_division=0))
print(f"Hamming Loss: {hamming_loss(y_true_issues, y_pred_issues)}")

In [ ]:
# 📈 Confidence Score Distribution (Calibration)
import matplotlib.pyplot as plt
import seaborn as sns

confidences = tf.reduce_max(tf.nn.softmax(model.predict(X_test_skin_type), axis=1), axis=1)
sns.histplot(confidences.numpy(), bins=20)
plt.title("Prediction Confidence Distribution")
plt.xlabel("Confidence")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# 🧪 Optional: Robustness Testing with Augmented Images
import albumentations as A
import cv2
import numpy as np

augment = A.Compose([
    A.RandomBrightnessContrast(p=1),
    A.MotionBlur(blur_limit=3, p=1),
])

# Example usage:
augmented_image = augment(image=cv2.imread('example.jpg'))['image']
# Display or evaluate on model
# plt.imshow(cv2.cvtColor(augmented_image, cv2.COLOR_BGR2RGB))
# plt.show()

## 📊 Additional Visualizations

### 🔷 Confusion Matrix Heatmap (Skin Type Classification)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Replace 'labels' with actual class names, e.g., ['Dry', 'Normal', 'Oily']
labels = ['Dry', 'Normal', 'Oily']
cm = confusion_matrix(y_true_type, y_pred_type_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix for Skin Type")
plt.show()

### 📉 ROC Curves for Multi-label Skin Issues

In [ ]:
from sklearn.metrics import roc_curve, auc

# y_pred_issues_prob: raw probabilities before thresholding (e.g., sigmoid outputs)
plt.figure(figsize=(10, 6))
for i in range(y_true_issues.shape[1]):
    fpr, tpr, _ = roc_curve(y_true_issues[:, i], y_pred_issues_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"Issue {i} (AUC = {roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Skin Issues")
plt.legend()
plt.grid(True)
plt.show()

### 📊 F1 Score per Skin Type Class

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

labels = ['Dry', 'Normal', 'Oily']
scores = precision_recall_fscore_support(y_true_type, y_pred_type_classes, zero_division=0)

plt.bar(labels, scores[2])  # F1-score per class
plt.ylabel("F1 Score")
plt.title("F1 Score per Skin Type")
plt.ylim(0, 1)
plt.show()

### 🖼️ Sample Predictions (Images and Labels)

In [ ]:
import matplotlib.pyplot as plt

for i in range(5):
    plt.imshow(X_test[i])
    plt.title(f"True: {y_true_type[i]} | Pred: {y_pred_type_classes[i]}")
    plt.axis('off')
    plt.show()

### 📈 Training vs Validation Loss Curves

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()